Vérifier que le dataset est bien accessible et print sa taille

In [2]:
from models.lcamazon import LCAmazon


dataset=LCAmazon(root="DATA", modality="s2", split="train")
print(f"dataset of length {len(dataset)}")

dataset of length 3840


Les images sont de taille 47x47 pixels et ont 12 bands

In [3]:
import numpy as np
img, label = dataset[0]
print(f"Image of shape: {np.shape(img)}, Label mask (ground truth) of shape: {np.shape(label)}")

Image of shape: (47, 47, 12), Label mask (ground truth) of shape: (47, 47)


Plot interactif des images satellites en RGB

In [3]:
# plot individual samples
from ipywidgets import interact
import matplotlib.pyplot as plt
@interact(idx=range(len(dataset)))
def plot_sample(idx=0):
    img, label = dataset[idx]
    red   = img[:, :, 3]   # Band 4
    green = img[:, :, 2]   # Band 3
    blue  = img[:, :, 1]   # Band 2
    #plt.imshow(img[:,:,3])
    rgb = np.stack([red, green, blue], axis=-1).astype(np.float32)

    # Normalize for display
    rgb /= np.percentile(rgb, 99)
    rgb = np.clip(rgb, 0, 1)

    # --- Plot ---
    plt.figure(figsize=(6, 6))
    plt.imshow(rgb)

interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

Plot interactif des ground truths avec la légende de quel classe est chaque pixel

In [4]:
# plot labels of samples
from ipywidgets import interact
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

@interact(idx=range(len(dataset)))
def plot_sample(idx=0):
    img, label = dataset[idx]

    class_mapping = {
        # mapping classes using the new indices from 1 to 12 instead of indices from the original dataset
        new_id: class_name
        for class_name, old_id in LCAmazon.LABEL_CLASSES.items()
        if old_id in LCAmazon.LABEL_REMAP
        for new_id in [LCAmazon.LABEL_REMAP[old_id]]
    }
    
    # Get unique labels present in this sample
    unique_labels = np.unique(label)

    # Create a colormap (one color per class)
    cmap = plt.cm.get_cmap("tab20", len(class_mapping))

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(label, cmap=cmap, vmin=0, vmax=len(class_mapping) - 1)
    ax.set_title("Ground Truth Segmentation")
    ax.axis("off")

    # Build legend patches (only for labels present)
    legend_patches = [
        mpatches.Patch(
            color=cmap(class_id),
            label=class_mapping[class_id]
        )
        for class_id in unique_labels
    ]

    # Place legend beside the image
    ax.legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        borderaxespad=0.
    )

    plt.tight_layout()
    plt.show()


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

Les deux ensembles, pour voir une image et son groundtruth de manière automatique (merge des 2 cells au-dessus)

In [4]:
# both cells above together
from ipywidgets import interact
import matplotlib.pyplot as plt
@interact(idx=range(len(dataset)))
def plot_sample(idx=0):
    # -------------- pretreatment for RGB ----------------------
    img, label = dataset[idx]
    red   = img[:, :, 3]   # Band 4
    green = img[:, :, 2]   # Band 3
    blue  = img[:, :, 1]   # Band 2
    #plt.imshow(img[:,:,3])
    rgb = np.stack([red, green, blue], axis=-1).astype(np.float32)

    # Normalize for display
    rgb /= np.percentile(rgb, 99)
    rgb = np.clip(rgb, 0, 1)

    # ------------------ pretreatment for groundtruth -------------------
    class_mapping = {
        # mapping classes using the new indices from 1 to 12 instead of indices from the original dataset
        new_id: class_name
        for class_name, old_id in LCAmazon.LABEL_CLASSES.items()
        if old_id in LCAmazon.LABEL_REMAP
        for new_id in [LCAmazon.LABEL_REMAP[old_id]]
    }

    # Get unique labels present in this sample
    unique_labels = np.unique(label)

    # Create a colormap (one color per class)
    cmap = plt.cm.get_cmap("tab20", len(class_mapping))

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    #RGB
    axes[0].imshow(rgb)
    axes[0].set_title("Satellite RGB Image")
    axes[0].axis("off")
    #GT
    axes[1].imshow(label, cmap=cmap, vmin=0, vmax=len(class_mapping) - 1)
    axes[1].set_title("Ground Truth Segmentation")
    axes[1].axis("off")
    #Legend
    legend_patches = [
        mpatches.Patch(
            color=cmap(class_id),
            label=class_mapping[class_id]
        )
        for class_id in unique_labels
    ]

    axes[1].legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        borderaxespad=0.
    )

    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

In [5]:
# Together with sentinel2 RGB image:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from ipywidgets import interact

REMAPPED_ID_TO_NAME = {
    LCAmazon.LABEL_REMAP[v]: k
    for k, v in LCAmazon.LABEL_CLASSES.items()
}

N_CLASSES = max(REMAPPED_ID_TO_NAME.keys())  # 12
CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)
PRED_ROOT = "modeloutputs/s2_prediction"

@interact(idx=range(len(dataset)))
def plot_all(idx=0):
    img, gt_label = dataset[idx]

    # --- Build RGB image ---
    red   = img[:, :, 3]   # Band 4
    green = img[:, :, 2]   # Band 3
    blue  = img[:, :, 1]   # Band 2

    rgb = np.stack([red, green, blue], axis=-1).astype(np.float32)
    rgb /= np.percentile(rgb, 99)
    rgb = np.clip(rgb, 0, 1)

    # --- Load prediction ---
    _, gt_path = dataset.samples[idx]
    fname = os.path.basename(gt_path)
    pred_path = os.path.join(PRED_ROOT, fname)

    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"Prediction not found: {pred_path}")

    with rasterio.open(pred_path) as src:
        pred_label = src.read(1).astype(np.int32)

    # --- Plot all three ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # RGB
    axes[0].imshow(rgb)
    axes[0].set_title("RGB Composite")
    axes[0].axis("off")

    # Ground Truth
    axes[1].imshow(gt_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    # Prediction
    axes[2].imshow(pred_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[2].set_title("Prediction")
    axes[2].axis("off")

    # --- Legend (union of GT + prediction labels) ---
    unique_labels = np.unique(
        np.concatenate([np.unique(gt_label), np.unique(pred_label)])
    )

    legend_patches = []
    for class_id in unique_labels:
        if class_id == 0:
            name = "Background / Ignored"
        else:
            name = REMAPPED_ID_TO_NAME.get(class_id, f"Unknown ({class_id})")

        legend_patches.append(
            mpatches.Patch(color=CMAP(class_id), label=name)
        )

    fig.legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 0.5),
        loc="center left"
    )

    plt.tight_layout()
    plt.show()


/tmp/ipykernel_1020463/483889935.py:15: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

NOW LET'S COMPARE OUR MODEL OUTPUT WITH GROUNDTRUTH (MODEL = SENTINEL2)

In [ ]:
import os
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from ipywidgets import interact

REMAPPED_ID_TO_NAME = {
    LCAmazon.LABEL_REMAP[v]: k
    for k, v in LCAmazon.LABEL_CLASSES.items()
}

N_CLASSES = max(REMAPPED_ID_TO_NAME.keys())  # 12
CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)
PRED_ROOT = "modeloutputs/s2_prediction"

@interact(idx=range(len(dataset)))
def plot_gt_vs_pred(idx=0):
    img, gt_label = dataset[idx]

    # Get filename from dataset
    _, gt_path = dataset.samples[idx]
    fname = os.path.basename(gt_path)

    pred_path = os.path.join(PRED_ROOT, fname)

    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"Prediction not found: {pred_path}")

    # Load prediction
    with rasterio.open(pred_path) as src:
        pred_label = src.read(1).astype(np.int32)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(gt_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")

    axes[1].imshow(pred_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[1].set_title("Prediction")
    axes[1].axis("off")

    # Legend (union of GT and prediction labels)
    unique_labels = np.unique(
        np.concatenate([np.unique(gt_label), np.unique(pred_label)])
    )

    legend_patches = []
    for class_id in unique_labels:
        if class_id == 0:
            name = "Background / Ignored"
        else:
            name = REMAPPED_ID_TO_NAME.get(class_id, f"Unknown ({class_id})")

        legend_patches.append(
            mpatches.Patch(color=CMAP(class_id), label=name)
        )

    fig.legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 0.5),
        loc="center left"
    )

    plt.tight_layout()
    plt.show()


NOW FOR THE MODEL BASED ON AE EMBEDDINGS (Maxime : still need to fix the function saving images to modeloutputs/AE_prediction)

In [7]:
import os
import rasterio
REMAPPED_ID_TO_NAME = {
    LCAmazon.LABEL_REMAP[v]: k
    for k, v in LCAmazon.LABEL_CLASSES.items()
}

N_CLASSES = max(REMAPPED_ID_TO_NAME.keys())  # 12
CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)
PRED_ROOT = "modeloutputs/AE_prediction"

@interact(idx=range(len(dataset)))
def plot_gt_vs_pred(idx=0):
    img, gt_label = dataset[idx]

    # Get filename from dataset
    _, gt_path = dataset.samples[idx]
    fname = os.path.basename(gt_path)

    pred_path = os.path.join(PRED_ROOT, fname)

    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"Prediction not found: {pred_path}")

    # Load prediction
    with rasterio.open(pred_path) as src:
        pred_label = src.read(1).astype(np.int32)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(gt_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")

    axes[1].imshow(pred_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[1].set_title("Prediction")
    axes[1].axis("off")

    # Legend (union of GT and prediction labels)
    unique_labels = np.unique(
        np.concatenate([np.unique(gt_label), np.unique(pred_label)])
    )

    legend_patches = []
    for class_id in unique_labels:
        if class_id == 0:
            name = "Background / Ignored"
        else:
            name = REMAPPED_ID_TO_NAME.get(class_id, f"Unknown ({class_id})")

        legend_patches.append(
            mpatches.Patch(color=CMAP(class_id), label=name)
        )

    fig.legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 0.5),
        loc="center left"
    )

    plt.tight_layout()
    plt.show()


/tmp/ipykernel_1020463/2445333875.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…